# California Housing Data Preprocessing

The notebook has two parts: **1. Functions** (define once) and **2. Run the pipeline** (one section per step).

# 1. FUNCTIONS

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split

TARGET = "median_house_value"
CATEGORICAL = "ocean_proximity"

### Reading and inspecting

In [2]:
def load_data(path):
    return pd.read_csv(path)


def information(df, rows=10):
    df.info()
    print("missing:", df.isnull().sum().sum(), "  duplicates:", df.duplicated().sum())
    print(df.describe())
    print(df.head(rows))

### Target and outliers

In [3]:
def split_target(df, target=TARGET):
    return df.drop(columns=target), df[target]


def iqr_bounds(s):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr


def outlier_mask(s):
    low, high = iqr_bounds(s)
    return (s < low) | (s > high)


def report_outliers(df, column=TARGET):
    low, high = iqr_bounds(df[column])
    print("Lower:", low, "  Upper:", high)
    print("Number of outliers:", outlier_mask(df[column]).sum())

### Cleaning and feature selection

In [4]:
def fill_missing_with_median(df, column):
    df = df.copy()
    df[column] = df[column].fillna(df[column].median())
    return df


def correlation_with_target(df, features, target=TARGET):
    return df[features].corrwith(df[target]).abs().sort_values(ascending=False)


def drop_features(df, columns):
    return df.drop(columns=columns)


def category_counts(df, column=CATEGORICAL):
    return df[column].value_counts()

### Encoding, splitting and scaling

In [5]:
def encode_categorical(df, column=CATEGORICAL):
    ohe = OneHotEncoder(drop="first", sparse_output=False)
    encoded_array = ohe.fit_transform(df[[column]])
    encoded_columns = ohe.get_feature_names_out([column])
    df_encoded = pd.DataFrame(encoded_array, columns=encoded_columns, index=df.index)
    return pd.concat([df.drop(columns=[column]), df_encoded], axis=1)


def split_and_scale(df, target=TARGET, test_size=0.2, random_state=42):
    X, y = split_target(df, target)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)   # fit on train only
    X_test_scaled = scaler.transform(X_test)
    return X_train, X_train_scaled, X_test_scaled, y_train, y_test

# 2. RUN THE PIPELINE

## READING THE CSV_FILE

In [6]:
df = load_data("housing.csv")
information(df)

<class 'pandas.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           20640 non-null  float64
 1   latitude            20640 non-null  float64
 2   housing_median_age  20640 non-null  float64
 3   total_rooms         20640 non-null  float64
 4   total_bedrooms      20433 non-null  float64
 5   population          20640 non-null  float64
 6   households          20640 non-null  float64
 7   median_income       20640 non-null  float64
 8   median_house_value  20640 non-null  float64
 9   ocean_proximity     20640 non-null  str    
dtypes: float64(9), str(1)
memory usage: 1.6 MB
missing: 207   duplicates: 0
          longitude      latitude  housing_median_age   total_rooms  \
count  20640.000000  20640.000000        20640.000000  20640.000000   
mean    -119.569704     35.631861           28.639486   2635.763081   
std        2.003532     

## DETERMINE THE TARGET FEATURE

In [7]:
x, y = split_target(df)
report_outliers(df)

Lower: -98087.5   Upper: 482412.5
Number of outliers: 1071


## DEALING WITH MISSING VALUES

In [8]:
df = fill_missing_with_median(df, "total_bedrooms")
print(df.isnull().sum())

longitude             0
latitude              0
housing_median_age    0
total_rooms           0
total_bedrooms        0
population            0
households            0
median_income         0
median_house_value    0
ocean_proximity       0
dtype: int64


## FINDING THE CORRELATION WITH EACH FEATURE

In [9]:
features = ['longitude', 'latitude', 'housing_median_age', 'total_rooms',
            'total_bedrooms', 'population', 'households', 'median_income']
print(correlation_with_target(df, features))

median_income         0.688075
latitude              0.144160
total_rooms           0.134153
housing_median_age    0.105623
households            0.065843
total_bedrooms        0.049457
longitude             0.045967
population            0.024650
dtype: float64


## FEATURE SELECTION

In [10]:
df = drop_features(df, ["population"])
print(df.head(10))
print(category_counts(df))

   longitude  latitude  housing_median_age  total_rooms  total_bedrooms  \
0    -122.23     37.88                41.0        880.0           129.0   
1    -122.22     37.86                21.0       7099.0          1106.0   
2    -122.24     37.85                52.0       1467.0           190.0   
3    -122.25     37.85                52.0       1274.0           235.0   
4    -122.25     37.85                52.0       1627.0           280.0   
5    -122.25     37.85                52.0        919.0           213.0   
6    -122.25     37.84                52.0       2535.0           489.0   
7    -122.25     37.84                52.0       3104.0           687.0   
8    -122.26     37.84                42.0       2555.0           665.0   
9    -122.25     37.84                52.0       3549.0           707.0   

   households  median_income  median_house_value ocean_proximity  
0       126.0         8.3252            452600.0        NEAR BAY  
1      1138.0         8.3014            

## ENCODING CATEGORICAL FEATURES

In [11]:
df = encode_categorical(df)
print(df.head())
print(df.info())

   longitude  latitude  housing_median_age  total_rooms  total_bedrooms  \
0    -122.23     37.88                41.0        880.0           129.0   
1    -122.22     37.86                21.0       7099.0          1106.0   
2    -122.24     37.85                52.0       1467.0           190.0   
3    -122.25     37.85                52.0       1274.0           235.0   
4    -122.25     37.85                52.0       1627.0           280.0   

   households  median_income  median_house_value  ocean_proximity_INLAND  \
0       126.0         8.3252            452600.0                     0.0   
1      1138.0         8.3014            358500.0                     0.0   
2       177.0         7.2574            352100.0                     0.0   
3       219.0         5.6431            341300.0                     0.0   
4       259.0         3.8462            342200.0                     0.0   

   ocean_proximity_ISLAND  ocean_proximity_NEAR BAY  \
0                     0.0            

## TRAIN/TEST SPLIT AND SCALING

In [12]:
X_train, X_train_scaled, X_test_scaled, y_train, y_test = split_and_scale(df)

print("Before scaling (first 5 training rows):")
print(X_train.head())
print("\nAfter scaling (same rows):")
print(pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index).head().round(2))

Before scaling (first 5 training rows):
       longitude  latitude  housing_median_age  total_rooms  total_bedrooms  \
14196    -117.03     32.71                33.0       3126.0           627.0   
8267     -118.16     33.77                49.0       3382.0           787.0   
17445    -120.48     34.66                 4.0       1897.0           331.0   
14265    -117.11     32.69                36.0       1421.0           367.0   
2271     -119.80     36.78                43.0       2382.0           431.0   

       households  median_income  ocean_proximity_INLAND  \
14196       623.0         3.2596                     0.0   
8267        756.0         3.8125                     0.0   
17445       336.0         4.1563                     0.0   
14265       355.0         1.9425                     0.0   
2271        380.0         3.5542                     1.0   

       ocean_proximity_ISLAND  ocean_proximity_NEAR BAY  \
14196                     0.0                       0.0   
8267  